In [ ]:
import os

import torch
import torch.nn.functional as F

import numpy as np
import xarray as xr
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

import rasterio as rio

import skimage as ski

from model import UNet10m
model_path = './checkpoints/pad-unet-ce-10m-max-pool/checkpoint_100.pth'

lf_dir = '/home/rdemilt/fuels/eo-fuels-extrapolation/data/LF2024_FBFM40.csv'

data_root = '/home/rdemilt/fuels/eo-fuels-extrapolation/data/fuels-tiles/'
pyrome = 27
tilenum = '00433'

FM40_LABELS = [
    91, 92, 93, 98, 99,
    101, 102, 103, 104, 105, 106, 107, 108, 109,
    121, 122, 123, 124,
    141, 142, 143, 144, 145, 146, 147, 148, 149,
    161, 162, 163, 164, 165,
    181, 182, 183, 184, 185, 186, 187, 188, 189,
    201, 202, 203, 204,
]

dropped_labels = [91,92,93,98,99]

valid_labels = [label for label in FM40_LABELS if all([label != drop for drop in dropped_labels])]

def make_label_encoder(valid_labels):
    label_map = dict(zip(valid_labels,np.arange(len(valid_labels))))
    label_map[-1] = -1
    

    def encode_fn(x):
        return label_map[x]
    
    label_encode = np.vectorize(encode_fn)
    
    return label_encode

label_encoder = make_label_encoder(valid_labels)

lfdf = pd.read_csv(lf_dir)
colors = dict(zip(lfdf['VALUE'],zip(lfdf['R'],lfdf['G'],lfdf['B'])))

In [2]:
model = UNet10m(
    n_channels=64,
    n_classes=40,
    in_sz=(480,480),
    out_sz=(160,160)
)
model.load_state_dict(torch.load(model_path,weights_only=True,map_location=torch.device('cpu')))
model.eval()

UNet10m(
  (inc): DoubleConv(
    (double_conv): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): Down(
    (maxpool_conv): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=Fals